In [ ]:
from agents import (
    Runner,
    function_tool,
    RunContextWrapper,
    Agent
)
from dataclasses import asdict, dataclass, field

In [42]:
@dataclass
class SectionContext:
    definition: str
    examples: str

In [43]:
@function_tool
async def after_definition(context: RunContextWrapper[SectionContext], definition_text: str, msg: str):
    section_context = context.context

    example_agent = Agent(
        name = "example_agent",
        instructions=f"Write examples in English for this topic: {definition_text}. Use these requirements: {msg}",
        tools = []
    )
    response = await Runner.run(example_agent, msg, context=section_context if section_context else context)
    
    section_context.examples = response.final_output

    return "Success"

@function_tool
async def start_writting(context: RunContextWrapper[SectionContext], msg: str):
    # Get the underlying SectionContext object.
    section_context = context.context
    
    definition_agent = Agent(
        name = "definition_agent",
        instructions="""You write definitions in English. Follow these steps:
        1. Write the definition.
        2. Call after_definition with the definition as definition_text and the requested exercise types as msg.
        3. Pass the full definition to after_definition.""",
        tools = [after_definition]
    )
    response = await Runner.run(definition_agent, msg, context=section_context if section_context else context)
    
    section_context.definition = response.final_output

    return "Success"


In [44]:
section_agent = Agent(
    name="section agent",
    instructions="You write a notebook section in English. Call start_writting directly and pass it the topic.",
    tools = [start_writting]
    )

In [45]:
sc = SectionContext(definition = "", examples = "")

result = await Runner.run(
                section_agent,
                "Write notes on the domain and range of a function.",
                context = sc,
            )

In [46]:
result.final_output

'Notes on the domain and range of a function are ready.'

In [47]:
print(sc)

SectionContext(definition='Domain and range are fundamental concepts in elementary mathematics.\n\nThe domain is the set of inputs for which the function is defined. For example, the domain of f(x) = 1/x is all real numbers except zero, because dividing by zero is undefined.\n\nThe range is the set of outputs a function takes on as the input varies over its domain. For example, if f(x) = x² is defined on all real numbers, its range is the set of nonnegative real numbers.\n\n- Domain: all allowed input values x\n- Range: all corresponding output values y\n\nThe following exercises apply these definitions.', examples='**Multiple-choice question**\n\nWhat is the domain of \\( f(x) = \\sqrt{x-2} \\)?\n\nA. \\( x > 0 \\)\nB. \\( x \\geq 2 \\)\nC. \\( x \\leq 2 \\)\nD. All real numbers\n\n---\n\n**Fill-in-the-blank question**\n\nIf the domain of \\( g(x) = 2x + 1 \\) is all real numbers, its range is ________.\n\n---\n\n**Answers**\n\nMultiple choice: B\nFill in the blank: All real numbers, 